# Pipeline Load Asteroids

This notebook is designed to be called from a Data Pipeline (Option B in Module 03).
It receives the NASA NeoWs API JSON response as a pipeline parameter and writes it
as a Delta table in the lakehouse.

**Pipeline parameter:** `api_response` (String) — the JSON response from the Web activity.

In [ ]:
import json
from pyspark.sql import Row

# Read the pipeline parameter passed via Base parameters
api_response_json = spark.conf.get("spark.synapse.notebook.pipeline.param.api_response", "{}")

data = json.loads(api_response_json)

# Flatten the nested JSON into rows
rows = []
for date, neos in data.get("near_earth_objects", {}).items():
    for neo in neos:
        rows.append(Row(
            neo_id=neo["id"],
            name=neo["name"],
            absolute_magnitude=float(neo.get("absolute_magnitude_h", 0)),
            is_hazardous=neo["is_potentially_hazardous_asteroid"],
            close_approach_date=date,
            miss_distance_km=float(neo["close_approach_data"][0]["miss_distance"]["kilometers"]),
            relative_velocity_kph=float(neo["close_approach_data"][0]["relative_velocity"]["kilometers_per_hour"]),
            estimated_diameter_min_m=float(neo["estimated_diameter"]["meters"]["estimated_diameter_min"]),
            estimated_diameter_max_m=float(neo["estimated_diameter"]["meters"]["estimated_diameter_max"])
        ))

df = spark.createDataFrame(rows)
df.write.mode("overwrite").format("delta").saveAsTable("lh_zosa.asteroids_bronze")
print(f"\u2705 Loaded {df.count()} asteroid records into asteroids_bronze")